# Урок 04 - Облик употребе алата

У овом лекцији ћете научити облик дизајна **Употреба алата** за AI агенте користећи Microsoft Agent Framework (Python). Обрађујемо:

- Дефинисање функција алата са `@tool` декоратером и типовним параметрима
- Обезбеђивање шема алата како би модел знао шта сваки алат ради
- Контролу извршења алата уз `approval_mode`
- Враћање **структурисаног излаза** преко Pydantic модела и `response_format`

Сценарио је **агент за резервације путовања** који може да претражује дестинације, проверава расположивост и преузима информације о летовима.


## Постављање


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated

from pydantic import BaseModel
from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Azure AI Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Дефинисање алата уз помоћ @tool декоратора

`@tool` декоратор претвара обичну Python функцију у алат који агент може позвати.  
Кључне појединости:

- **Документујући низ** постаје опис алата који модел види.  
- **Антоације типова** (укључујући `Annotated` са описима) дефинишу шему алата.  
- `approval_mode` контролише да ли корисник мора одобрити сваки позив пре него што се изврши.


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get available vacation destinations."""
    return ["Barcelona", "Paris", "Berlin", "Tokyo", "Sydney", "New York City"]


@tool(approval_mode="never_require")
def check_availability(
    destination: Annotated[str, "The destination to check"],
) -> str:
    """Check booking availability for a destination."""
    availability = {
        "Barcelona": "Available - 3 spots left",
        "Paris": "Available",
        "Berlin": "Sold out",
        "Tokyo": "Available - 1 spot left",
        "Sydney": "Available",
        "New York City": "Available",
    }
    return availability.get(destination, "Unknown destination")


@tool(approval_mode="never_require")
def get_flight_info(
    origin: Annotated[str, "Origin airport code"],
    destination: Annotated[str, "Destination airport code"],
) -> str:
    """Get flight information between two cities."""
    flights = {
        "LHR-BCN": "BA 2042, Departs 08:30, Arrives 11:45, $350",
        "LHR-CDG": "AF 1081, Departs 09:15, Arrives 11:30, $280",
        "LHR-NRT": "JL 044, Departs 11:00, Arrives 07:00+1, $890",
    }
    return flights.get(
        f"{origin}-{destination}",
        f"No direct flights from {origin} to {destination}",
    )

## Креирање агента са више алата

Проследите сва три алата клијенту како би модел могао да позове оне које су му потребне да одговори на корисниково питање.


In [ ]:
travel_tools = [get_destinations, check_availability, get_flight_info]

agent = client.as_agent(
    name="TravelToolAgent",
    instructions="You are a travel agent. Use the available tools to answer questions about destinations, availability, and flights.",
    tools=travel_tools,
)

response = await agent.run(
    "What destinations do you have? Which ones are still available?"
)
print(response)

## Структурирани излаз са алатима

Постављањем `response_format` на Пјидантиц модел, агент је приморан да врати добро типизован JSON објекат уместо текста слободног облика. Ово је корисно када downstream код треба да програмски користи резултат.


In [ ]:
class BookingRecommendation(BaseModel):
    destination: str
    available: bool
    flight_details: str
    estimated_cost: int


class TravelPlan(BaseModel):
    recommendations: list[BookingRecommendation]


structured_agent = client.as_agent(
    name="StructuredTravelAgent",
    instructions=(
        "You are a travel agent. Use the available tools to find destinations, "
        "check availability, and get flight info. Return structured results."
    ),
    tools=[get_destinations, check_availability, get_flight_info],
)

response = await structured_agent.run(
    "I want to fly from London Heathrow to somewhere warm in Europe. "
    "Check what's available."
)
if response:
    print(response)

## Обрасци одобрења алата

Параметар `approval_mode` на `@tool` контролише да ли позиви алата захтевају људско одобрење пре извршавања:

| Режим | Понашање |
|---|---|
| `"never_require"` | Алат се покреће аутоматски — није потребна потврда корисника. |
| `"always_require"` | Сваки позив мора бити одобрен од стране корисника пре извршавања. |

Користите `"always_require"` за алате који имају споредне ефекте (нпр. резервација лета, наплата кредитне картице) како би људи остали укључени у процес.


In [ ]:
@tool(approval_mode="always_require")
def book_flight(
    origin: Annotated[str, "Origin airport code"],
    destination: Annotated[str, "Destination airport code"],
    passenger_name: Annotated[str, "Full name of the passenger"],
) -> str:
    """Book a flight for a passenger. Requires approval before executing."""
    return (
        f"Flight booked from {origin} to {destination} "
        f"for {passenger_name}. Confirmation #TRV-2024-{hash(passenger_name) % 10000:04d}"
    )


print("Tool name:", book_flight.name)
print("Approval mode:", book_flight.approval_mode)

## Сажетак

У овој лекцији сте научили како да:

1. **Дефинишете алате** користећи `@tool` декоратор са типизираним параметрима и документацијом која служи као шема алата.
2. **Комбинујете више алата** тако да агент може да их позива у низу како би одговорио на сложене упите.
3. **Вратите структурирани излаз** прослеђујући Пидантиц модел као `response_format`.
4. **Контролишете одобрење алата** са `approval_mode` да бисте задржали човека у ланцу за осетљиве операције.

Ови обрасци представљају основу за изградњу поузданих агената спремних за продукцију који могу безбедно да комуницирају са спољним системима.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Изјава о одрицању одговорности**:
Овај документ је преведен коришћењем услуге за аутоматски превод [Co-op Translator](https://github.com/Azure/co-op-translator). Иако тежимо тачности, имајте у виду да аутоматски преводи могу садржати грешке или нетачности. Оригинални документ на његовом изворном језику треба сматрати ауторитативним извором. За критичне информације препоручује се професионални људски превод. Нисмо одговорни за било каква неспоразума или погрешна тумачења која произилазе из коришћења овог превода.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
